# Amazon ML Challenge 2026 — Entity Resolution Pipeline

**Orchestration notebook** — connects every phase of the pipeline.

Strategy document: [ER_Strategy.md](ER_Strategy.md)

| Phase | Description | Where | Status |
|---|---|---|---|
| 0 | Setup & dependencies | Notebook (free) | Run once |
| 1 | Data Exploration | Notebook (free) | Done (view_explore.ipynb) |
| 2 | Normaliser smoke-test | Notebook (free) | |
| 3 | BGE-m3 Embeddings | AWS g4dn.4xlarge (paid) | |
| 4 | FAISS build + TF-IDF supplement | AWS + Notebook | |
| 5 | Feature Engineering | Notebook (free) | |
| 6 | LightGBM Stage 1 | Notebook (free) | |
| 7 | Reranker fine-tuning | AWS g4dn.xlarge (paid) | |
| 8 | Test Inference + Submission | Notebook (free) | |
| 9 | Triple Validation | Notebook (free) | |

**Run cells in order top-to-bottom.** AWS Training Job cells (Phases 3, 4, 7) require
SageMaker credentials and incur GPU cost (~$22.89 total from $200 credits).

---
## Phase 0 — Setup

In [ ]:
# Install all dependencies (ER_Strategy.md §11)
!pip install -q lightgbm>=4.0 faiss-cpu>=1.7.4 sentence-transformers>=2.7 \
    transformers>=4.40 torch>=2.1 rapidfuzz>=3.0 pandas>=2.0 numpy>=1.24 \
    scikit-learn>=1.3 ftfy>=6.1 "indic-transliteration>=2.3" \
    boto3>=1.34 sagemaker>=2.200 pyarrow fastparquet

print('Dependencies installed.')

In [ ]:
import os
import sys
import logging
from pathlib import Path

# Ensure this notebook's directory is on the Python path so local modules resolve
NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# Paths — adjust if running from SageMaker (S3 paths differ)
DATA_DIR = '../student_resource/dataset'   # parent of train/ and test/
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('main')

print(f'Notebook dir : {NOTEBOOK_DIR}')
print(f'Data dir     : {Path(DATA_DIR).resolve()}')
print(f'Output dir   : {Path(OUTPUT_DIR).resolve()}')

---
## Phase 1 — Data Exploration (already done)

Full findings are in [ER Pipeline — Data Exploration Findings (Q26).md](ER%20Pipeline%20%E2%80%94%20Data%20Exploration%20Findings%20%28Q26%29.md).

Key confirmed facts (ER_Strategy.md §2F):
- train 12,527,040 records | test 11,702,133 records | grand total ~24.2M
- Singleton ratio 5.58% (123,247 unmatchable S1 entities)
- France is zero-shot (test only, 1,694,445 records)
- S1 train 100% Latin; S1 test 40,789 French accented-Latin (risk closed)
- S2/S3 ~15% non-Latin (Indian scripts — BGE-m3 is the only blocker)
- ~4% domain names, ~3.3% null addresses (all in S2/S3)

To re-run exploration:

In [ ]:
# Uncomment to re-run data exploration
# %run view_explore.ipynb

---
## Phase 2 — Normaliser Smoke-test

In [ ]:
import normaliser
from normaliser import normalise_record, canonical_country

# Quick sanity-check — 5 representative records from the noise audit (ER_Strategy §2F)
test_cases = [
    # (entity_id, name, address, country)
    ('T1', 'limited sarasva india', '9/1/3. KASUNDIA 2ND BYE LANE', 'India'),
    ('T2', 'Lambodar Sfocwares', '', 'India'),   # null address
    ('T3', 'PRAIRIE CAPITAL PLLC-PARTNERS', '120. Autumn Woods Boulevard', 'US'),
    ('T4', 'cardiologymetrocare.com', '1138 TRIMBLE CREEK DRIVE', 'US'),
    ('T5', 'Àmicale Des Anciens Combattants', '75001 Paris Cedex 1', 'France'),
    # A34 critical test — hyphenated Indian house number must be preserved
    ('T6', 'My Construction Private Limited', 'H.No 1-1-742/3, Gandhi Nagar', 'India'),
]

print(f'{'ID':<5} {'name_core':<35} {'addr_tokens (sample)':<40} {'addr_is_null'}')
print('-' * 100)
for eid, name, addr, country in test_cases:
    rec = normalise_record(entity_id=eid, business_name=name,
                           business_address=addr, country=country)
    addr_sample = str(sorted(rec.addr_tokens)[:5])
    print(f'{eid:<5} {rec.name_core:<35} {addr_sample:<40} {rec.addr_is_null}')

# A34 critical assertion
t6 = normalise_record('T6', 'My Construction Private Limited', 'H.No 1-1-742/3, Gandhi Nagar', 'India')
# The hyphenated number 1-1-742 (or its components) must appear in addr_tokens
addr_joined = ' '.join(t6.addr_tokens)
assert any(t in addr_joined for t in ['1-1-742', '742', '1-1']), \
    f'A34 FAIL — hyphenated number destroyed. addr_tokens: {t6.addr_tokens}'
print('\nA34 assertion PASSED — hyphenated Indian house number preserved.')

---
## Phase 3 — BGE-m3 Embeddings (AWS Training Jobs)

These cells launch SageMaker Training Jobs on `ml.g4dn.4xlarge` (T4 16GB VRAM).
Budget: ~$8.73 (Job 1 train) + ~$8.14 (Job 2 test) = ~$16.87 from $200 credits.

**Pre-requisites:**
- AWS credentials configured (IAM role with SageMaker + S3 access)
- S3 bucket created: `s3://er-challenge-<your-name>/`
- TSV files uploaded to S3

Output: `.npy` embedding shards (500K records each) uploaded to S3 automatically.

In [ ]:
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch

# ---- Configure these once ------------------------------------------------
S3_BUCKET = 'er-challenge-<your-name>'   # replace with your bucket name
S3_PREFIX = 'er-challenge'
REGION    = 'us-east-1'
# --------------------------------------------------------------------------

sess    = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))
role    = sagemaker.get_execution_role()
s3_data = f's3://{S3_BUCKET}/{S3_PREFIX}/data'

print(f'SageMaker role: {role}')
print(f'S3 data prefix: {s3_data}')

In [ ]:
# Job 1 — Embed train set (train_source1/2/3.tsv)
embed_job_train = PyTorch(
    entry_point='train_embeddings.py',
    source_dir='.',          # uploads the whole Experiment_Day1/ folder
    role=role,
    framework_version='2.1',
    py_version='py310',
    instance_type='ml.g4dn.4xlarge',
    instance_count=1,
    hyperparameters={
        'job-mode': 'train',
        'batch-size': 256,
        'shard-size': 500000,
        'fp16': '',
    },
    output_path=f's3://{S3_BUCKET}/{S3_PREFIX}/embeddings',
    base_job_name='er-embed-train',
)
embed_job_train.fit({'data': s3_data}, wait=False)
print(f'Job 1 launched: {embed_job_train.latest_training_job.name}')

In [ ]:
# Job 2 — Embed test set (test_source1/2/3.tsv)
embed_job_test = PyTorch(
    entry_point='train_embeddings.py',
    source_dir='.',
    role=role,
    framework_version='2.1',
    py_version='py310',
    instance_type='ml.g4dn.4xlarge',
    instance_count=1,
    hyperparameters={
        'job-mode': 'test',
        'batch-size': 256,
        'shard-size': 500000,
        'fp16': '',
    },
    output_path=f's3://{S3_BUCKET}/{S3_PREFIX}/embeddings',
    base_job_name='er-embed-test',
)
embed_job_test.fit({'data': s3_data}, wait=False)
print(f'Job 2 launched: {embed_job_test.latest_training_job.name}')

In [ ]:
# Wait for both jobs and download embedding shards to local output/embeddings/
import time

for job in [embed_job_train, embed_job_test]:
    job_name = job.latest_training_job.name
    print(f'Waiting for {job_name}...')
    job.latest_training_job.wait()
    print(f'{job_name} COMPLETE')

# Download shards from S3
!aws s3 sync s3://{S3_BUCKET}/{S3_PREFIX}/embeddings/ output/embeddings/
print('Embedding shards downloaded.')

---
## Phase 4 — FAISS Build + Blocking (AWS) + TF-IDF Supplement (Notebook)

In [ ]:
# Phase 4a — FAISS index build + candidate pairs (AWS g4dn.4xlarge)
# This job needs 64GB RAM for the FAISS index — ml.t3.medium cannot hold it.

faiss_job = PyTorch(
    entry_point='build_faiss_and_block.py',
    source_dir='.',
    role=role,
    framework_version='2.1',
    py_version='py310',
    instance_type='ml.g4dn.4xlarge',
    instance_count=1,
    hyperparameters={
        'job-mode': 'train',
        'top-k': 20,
        'nlist': 4096,
        'nprobe': 64,
        'eval-recall': '',
    },
    output_path=f's3://{S3_BUCKET}/{S3_PREFIX}/candidates',
    base_job_name='er-faiss-train',
)
faiss_job.fit({'data': f's3://{S3_BUCKET}/{S3_PREFIX}/embeddings'}, wait=True)
print('FAISS job complete.')

# Download candidate pairs
!aws s3 sync s3://{S3_BUCKET}/{S3_PREFIX}/candidates/ output/
print('Candidate pairs downloaded.')

In [ ]:
# Phase 4b — TF-IDF supplement (Notebook CPU)
import tfidf_blocking

FAISS_TRAIN = f'{OUTPUT_DIR}/candidate_pairs_train.tsv'
TFIDF_TRAIN = f'{OUTPUT_DIR}/candidate_pairs_train_with_tfidf.tsv'

merged_train = tfidf_blocking.run(
    split='train',
    data_dir=DATA_DIR,
    faiss_candidates_path=FAISS_TRAIN,
    output_path=TFIDF_TRAIN,
    top_k=20,
)
print(f'Train candidates after TF-IDF union: {len(merged_train):,}')

In [ ]:
# Measure blocking recall on train validation split (20% holdout)
# Denominator = matchable S1 only (2,083,574) NOT total S1 (2,206,821)
import pandas as pd

gt = pd.read_csv(f'{DATA_DIR}/train/train_ground_truth.tsv', sep='\t', dtype=str)
gt.columns = [c.strip().lower() for c in gt.columns]
s1_col, match_col = gt.columns[0], gt.columns[1]

# Build ground-truth pair set
gt_pairs = set()
matchable_s1 = set()
for _, row in gt.iterrows():
    s1_id = str(row[s1_col])
    for mid in str(row[match_col]).split(','):
        mid = mid.strip()
        if mid:
            gt_pairs.add((mid, s1_id))  # (query_id, s1_id)
            matchable_s1.add(s1_id)

# Check recall
candidates = merged_train[['query_entity_id', 'candidate_s1_entity_id']]
candidate_set = set(zip(candidates['query_entity_id'], candidates['candidate_s1_entity_id']))

recovered = gt_pairs & candidate_set
blocking_recall = len({s1 for _, s1 in recovered}) / len(matchable_s1)

print(f'Matchable S1 entities : {len(matchable_s1):,} (denominator)')
print(f'GT pairs total        : {len(gt_pairs):,}')
print(f'GT pairs recovered    : {len(recovered):,}')
print(f'Blocking recall       : {blocking_recall:.4f} ({blocking_recall*100:.2f}%)')

if blocking_recall < 0.95:
    print('WARNING: Recall < 95% target. Consider adding token/Soundex blocking keys (Round 2/3 in ER_Strategy §3).')
else:
    print('Recall >= 95% target. Blocking is sufficient — proceed to Phase 5.')

In [ ]:
# Same TF-IDF supplement for the test split
FAISS_TEST = f'{OUTPUT_DIR}/candidate_pairs_test.tsv'
TFIDF_TEST = f'{OUTPUT_DIR}/candidate_pairs_test_with_tfidf.tsv'

merged_test = tfidf_blocking.run(
    split='test',
    data_dir=DATA_DIR,
    faiss_candidates_path=FAISS_TEST,
    output_path=TFIDF_TEST,
    top_k=20,
)
print(f'Test candidates after TF-IDF union: {len(merged_test):,}')

---
## Phase 5 — Feature Engineering

In [ ]:
import feature_engineering

FEATURES_TRAIN = f'{OUTPUT_DIR}/features_train.parquet'
GT_PATH = f'{DATA_DIR}/train/train_ground_truth.tsv'

train_feats = feature_engineering.run(
    split='train',
    data_dir=DATA_DIR,
    candidates_path=TFIDF_TRAIN,
    output_path=FEATURES_TRAIN,
    ground_truth_path=GT_PATH,
    fit_tfidf=True,
)

print(f'Train feature matrix: {train_feats.shape}')
if 'label' in train_feats.columns:
    pos = train_feats['label'].sum()
    print(f'Positives: {pos:,} ({pos/len(train_feats)*100:.2f}%)')
train_feats.head(3)

In [ ]:
# Audit feature distributions (NaN check)
import pandas as pd
from train_lightgbm import FEATURE_COLS

print('=== Feature NaN audit ===')
nan_counts = train_feats[FEATURE_COLS].isna().sum()
if nan_counts.sum() == 0:
    print('No NaN values in any feature column.')
else:
    print(nan_counts[nan_counts > 0])

print('\n=== Feature summary statistics ===')
print(train_feats[FEATURE_COLS].describe().round(4).to_string())

In [ ]:
# Test split features (no labels)
FEATURES_TEST = f'{OUTPUT_DIR}/features_test.parquet'

test_feats = feature_engineering.run(
    split='test',
    data_dir=DATA_DIR,
    candidates_path=TFIDF_TEST,
    output_path=FEATURES_TEST,
    fit_tfidf=True,
)
print(f'Test feature matrix: {test_feats.shape}')

---
## Phase 6 — LightGBM Stage 1

In [ ]:
import train_lightgbm

lgbm_results = train_lightgbm.run(
    features_path=FEATURES_TRAIN,
    output_dir=OUTPUT_DIR,
    t_low=0.60,
    t_high=0.90,
    val_frac=0.20,
)

print('\n=== LightGBM Results ===')
print(f"Eval 1 F_0.5 (val)      : {lgbm_results['val_f05']:.4f}")
if lgbm_results.get('india_f05'):
    print(f"Eval 2 F_0.5 (India)    : {lgbm_results['india_f05']:.4f}")
print(f"Optimal threshold       : {lgbm_results['t_opt']:.2f}")
print(f"T_low / T_high used     : {lgbm_results['t_low_used']} / {lgbm_results['t_high_used']}")
print(f"ACCEPT (Stage-1 match)  : {lgbm_results['accept_count']:,}")
print(f"UNCERTAIN (to reranker) : {lgbm_results['uncertain_count']:,}")
print(f"REJECT                  : {lgbm_results['reject_count']:,}")

In [ ]:
# Threshold sweep visualisation
import matplotlib.pyplot as plt

sweep = lgbm_results['sweep_df']
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sweep['threshold'], sweep['F_0.5'], label='F_0.5', color='royalblue', linewidth=2)
ax.plot(sweep['threshold'], sweep['precision'], label='Precision', color='tomato', linestyle='--')
ax.plot(sweep['threshold'], sweep['recall'], label='Recall', color='seagreen', linestyle=':')
ax.axvline(lgbm_results['t_opt'], color='purple', linestyle='-', linewidth=1.5,
           label=f"T_opt={lgbm_results['t_opt']:.2f}")
ax.axvline(0.60, color='orange', linestyle='--', linewidth=1, label='T_low=0.60')
ax.axvline(0.90, color='red', linestyle='--', linewidth=1, label='T_high=0.90')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold sweep — Validation set')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/threshold_sweep.png', dpi=120)
plt.show()

In [ ]:
# Feature importance
fi = lgbm_results['feature_importance']
print('Top-15 features by gain:')
print(fi.head(15).to_string(index=False))

---
## Phase 7 — Reranker Fine-tuning (AWS Training Job)

Launches `train_reranker.py` on `ml.g4dn.xlarge` (T4 16GB VRAM).
Budget: ~$4.52 (3hr).

In [ ]:
# Upload uncertain pairs and source TSVs to S3 for the reranker job
!aws s3 cp {OUTPUT_DIR}/uncertain_pairs.tsv s3://{S3_BUCKET}/{S3_PREFIX}/reranker/uncertain_pairs.tsv
!aws s3 sync {DATA_DIR}/train/ s3://{S3_BUCKET}/{S3_PREFIX}/data/train/ --exclude '*.DS_Store'
print('Uploaded reranker inputs to S3.')

In [ ]:
# Launch reranker fine-tuning job
reranker_job = PyTorch(
    entry_point='train_reranker.py',
    source_dir='.',
    role=role,
    framework_version='2.1',
    py_version='py310',
    instance_type='ml.g4dn.xlarge',
    instance_count=1,
    hyperparameters={
        'max-positives': 300000,
        'neg-per-pos': 3,
        'hard-neg-frac': 0.8,
        'hard-neg-cosine': 0.7,
        'batch-size': 16,
        'epochs': 2,
        'lr': 2e-5,
        'fp16': '',
    },
    output_path=f's3://{S3_BUCKET}/{S3_PREFIX}/reranker_output',
    model_uri=None,
    base_job_name='er-reranker',
)
reranker_job.fit(
    {'data': f's3://{S3_BUCKET}/{S3_PREFIX}/reranker'},
    wait=True,
)
print('Reranker job complete.')

In [ ]:
# Download fine-tuned reranker model
!aws s3 sync s3://{S3_BUCKET}/{S3_PREFIX}/reranker_output/ output/reranker_model/
RERANKER_MODEL_DIR = f'{OUTPUT_DIR}/reranker_model'
print(f'Reranker model downloaded to {RERANKER_MODEL_DIR}')

---
## Phase 8 — Test Inference + Submission Generation

In [ ]:
# Stage 1 — Score test features with LightGBM
import train_lightgbm

T_LOW  = lgbm_results['t_low_used']
T_HIGH = lgbm_results['t_high_used']
LGBM_MODEL = lgbm_results['model_path']

test_scored = train_lightgbm.predict(
    features_path=FEATURES_TEST,
    model_path=LGBM_MODEL,
    t_low=T_LOW,
    t_high=T_HIGH,
    output_dir=OUTPUT_DIR,
)
print(test_scored['disposition'].value_counts())

In [ ]:
# Stage 2 — Reranker on uncertain test pairs
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RERANKER_MODEL_DIR = f'{OUTPUT_DIR}/reranker_model'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Reranker device: {DEVICE}')

tokenizer_ce = AutoTokenizer.from_pretrained(RERANKER_MODEL_DIR)
model_ce = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_DIR, num_labels=1
).to(DEVICE)
model_ce.eval()

uncertain_test = test_scored[test_scored['disposition'] == 'UNCERTAIN'].copy()
print(f'Uncertain pairs to rerank: {len(uncertain_test):,}')

In [ ]:
# Rerank uncertain test pairs
import torch.nn.functional as F
from normaliser import normalise_record

# Load test source TSVs into a lookup dict once
import pandas as pd

def _load_lookup(split_dir, split):
    lookup = {}
    for src in ['1', '2', '3']:
        path = f'{split_dir}/{split}_source{src}.tsv'
        df = pd.read_csv(path, sep='\t', dtype=str, low_memory=False)
        df.columns = [c.strip().lower() for c in df.columns]
        id_col = df.columns[0]
        name_col = next((c for c in df.columns if 'name' in c), None)
        addr_col = next((c for c in df.columns if 'address' in c or 'addr' in c), None)
        country_col = next((c for c in df.columns if 'country' in c), None)
        for _, row in df.iterrows():
            eid = str(row[id_col])
            if eid not in lookup:
                lookup[eid] = {
                    'name': str(row[name_col]) if name_col else '',
                    'addr': str(row[addr_col]) if addr_col else '',
                    'country': str(row[country_col]) if country_col else '',
                }
    return lookup

test_lookup = _load_lookup(f'{DATA_DIR}/test', 'test')

def _build_text(entity_id, lookup):
    info = lookup.get(entity_id, {})
    rec = normalise_record(
        entity_id=entity_id,
        business_name=info.get('name', ''),
        business_address=info.get('addr', ''),
        country=info.get('country', ''),
    )
    if rec.addr_is_null:
        return f'Business: {rec.name_normalized}'
    return f'Business: {rec.name_normalized}  Address: {rec.addr_normalized}'

# Threshold for reranker (tune on val set if time allows)
T_CE = 0.5
BATCH_SIZE_CE = 32

ce_probs = []
for i in range(0, len(uncertain_test), BATCH_SIZE_CE):
    batch = uncertain_test.iloc[i:i + BATCH_SIZE_CE]
    text_a = [_build_text(r['query_entity_id'], test_lookup) for _, r in batch.iterrows()]
    text_b = [_build_text(r['candidate_s1_entity_id'], test_lookup) for _, r in batch.iterrows()]
    enc = tokenizer_ce(text_a, text_b, padding=True, truncation=True,
                       max_length=512, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        logits = model_ce(**enc).logits.squeeze(-1)
    probs = torch.sigmoid(logits).cpu().numpy().tolist()
    ce_probs.extend(probs)

uncertain_test = uncertain_test.copy()
uncertain_test['p_ce'] = ce_probs
uncertain_test['reranker_match'] = (uncertain_test['p_ce'] > T_CE).astype(int)
print(f'Reranker matches: {uncertain_test["reranker_match"].sum():,} / {len(uncertain_test):,}')

In [ ]:
# Merge Stage-1 accepts and Stage-2 reranker matches into final matching_results.tsv
import pandas as pd

# Stage-1 accepted pairs
s1_matches = test_scored[
    test_scored['disposition'] == 'ACCEPT'
][['query_entity_id', 'candidate_s1_entity_id']].copy()

# Stage-2 reranker accepted pairs
s2_matches = uncertain_test[
    uncertain_test['reranker_match'] == 1
][['query_entity_id', 'candidate_s1_entity_id']].copy()

all_matches = pd.concat([s1_matches, s2_matches], ignore_index=True)
all_matches = all_matches.drop_duplicates()
print(f'Total matched pairs: {len(all_matches):,}')

# Aggregate: for each S1 entity, collect all matched S2/S3 query IDs
grouped = (
    all_matches
    .groupby('candidate_s1_entity_id')['query_entity_id']
    .apply(lambda x: ','.join(sorted(set(x))))
    .reset_index()
    .rename(columns={
        'candidate_s1_entity_id': 'source1_entity_id',
        'query_entity_id': 'matched_entity_ids',
    })
)

# Ensure every test S1 entity has a row (empty string = no match)
test_s1_df = pd.read_csv(
    f'{DATA_DIR}/test/test_source1.tsv', sep='\t', dtype=str, low_memory=False
)
test_s1_df.columns = [c.strip().lower() for c in test_s1_df.columns]
all_s1_ids = pd.DataFrame({'source1_entity_id': test_s1_df.iloc[:, 0].astype(str)})

submission = all_s1_ids.merge(grouped, on='source1_entity_id', how='left')
submission['matched_entity_ids'] = submission['matched_entity_ids'].fillna('')

os.makedirs(f'{OUTPUT_DIR}', exist_ok=True)
MATCHING_PATH = f'{OUTPUT_DIR}/matching_results.tsv'
submission.to_csv(MATCHING_PATH, sep='\t', index=False)
print(f'Saved matching_results.tsv: {len(submission):,} rows')

In [ ]:
# Save candidate_pairs.tsv for submission zip
# Required format: source1_entity_id | candidate_entity_ids (comma-sep)
cand_grouped = (
    merged_test
    .groupby('candidate_s1_entity_id')['query_entity_id']
    .apply(lambda x: ','.join(sorted(set(x))))
    .reset_index()
    .rename(columns={
        'candidate_s1_entity_id': 'source1_entity_id',
        'query_entity_id': 'candidate_entity_ids',
    })
)
cand_out = all_s1_ids.merge(cand_grouped, on='source1_entity_id', how='left')
cand_out['candidate_entity_ids'] = cand_out['candidate_entity_ids'].fillna('')
CANDIDATE_PATH = f'{OUTPUT_DIR}/candidate_pairs.tsv'
cand_out.to_csv(CANDIDATE_PATH, sep='\t', index=False)
print(f'Saved candidate_pairs.tsv: {len(cand_out):,} rows')

In [ ]:
# Validate submission before uploading
!python validate_submission.py \
    --matching {OUTPUT_DIR}/matching_results.tsv \
    --candidate {OUTPUT_DIR}/candidate_pairs.tsv \
    --test-dir {DATA_DIR}/test

---
## Phase 9 — Triple Validation (ER_Strategy §9)

In [ ]:
# Eval 1 summary (already computed in Phase 6)
print('=== Eval 1: Stratified 80/20 validation ===')
print(f"F_0.5 : {lgbm_results['val_f05']:.4f}")
print(f"Optimal threshold : {lgbm_results['t_opt']:.2f}")

# Eval 2 summary
if lgbm_results.get('india_f05'):
    print('\n=== Eval 2: US-train / India-validate ===')
    india_f = lgbm_results['india_f05']
    print(f"India F_0.5 : {india_f:.4f}")
    if india_f >= 0.80:
        print('France generalisation proxy: LIKELY OK (>= 0.80)')
    elif india_f >= 0.70:
        print('France generalisation proxy: MARGINAL (0.70-0.80) — monitor submission score')
    else:
        print('France generalisation proxy: POOR (< 0.70) — fix pipeline before submitting')

In [ ]:
# Eval 3 note: full test inference is the Eval 3 run — already done in Phase 8.
# The leaderboard score IS Eval 3.
print('=== Eval 3: Full pipeline on test set ===')
print(f'Output file : {MATCHING_PATH}')
print('Upload matching_results.tsv + candidate_pairs.tsv to the leaderboard to get Eval 3 F_0.5.')

# Quick stats
non_empty = (submission['matched_entity_ids'] != '').sum()
print(f'S1 entities with at least one match: {non_empty:,} / {len(submission):,}')
print(f'Match rate: {non_empty/len(submission)*100:.2f}%')

---
## Billing Reminders (ER_Strategy §2E)

- **Stop, don't delete** your SageMaker notebook when done for the day.
- **Delete any deployed endpoints** immediately (charge $0.12/hr even idle).
- Total GPU budget for this pipeline: ~$22.89 out of $200 credits.
- Set a billing alert in AWS Console if you haven't already.